# N4 · AOEI vs LETKF vs tempering, Nt = 1 … 5

N3 asked *where* assimilation helps. This notebook asks *which method* helps most,
and whether the answer survives changing the metric, the reduction, or the
variable being scored.

### The consistency contract with N3

N3 and N4 read the same files through the same loader, intersect the same points
with the same `align()`, and aggregate with the same `skill_summary()`. Cell 5
below re-derives N3's headline table and compares it two ways — against the digest
literal pasted from N3, and against the CSV N3 published. **If they differ, this
notebook raises and stops.** That is deliberate: a disagreement between the two
notebooks is a bug, not a finding.

### What is being compared

| method | what it does |
|---|---|
| LETKF | one analysis step at the nominal observation error |
| AOEI | inflates R where the departure is too large for the ensemble to explain, then one step |
| TEnKF Nt=k | k sequential steps with back-loaded inflation summing to the full observation weight |

LETKF should be numerically identical to TEnKF Nt=1 — same maths, different code
path. Cell 10 tests that rather than assuming it.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nbcommon as nb
nb.banner()
nb.assert_convention()

# ── configuration: IDENTICAL to N3 cell 2, deliberately ────────────────────
# Point these at the two sweep output directories under data/ once the runs land.
# `tag` is the directory name; `loc` is the isotropic localization scale it used.
# One config per scale -- loc_x/loc_y/loc_z are a Cartesian product in
# _build_combos, so a single config listing both would give 8 mixed-anisotropy
# combos. N4 must repeat this literal verbatim; the cross-check verifies it.
RUNS = [dict(tag="WS_sweep_L0.1", loc=0.1, hour="18"),
        dict(tag="WS_sweep_L2.0", loc=2.0, hour="18")]

METRICS    = ("rmse", "crps")
REF_METHOD = ("TEnKF", 2)

# Pasted from N3's final cell.
N3_DIGEST = None          # <-- set me from N3's output

MO_TAG = "WS_multiobs_L0.1"        # optional multi-obs run; skipped if absent

In [ ]:
COLS = nb.columns_for(metrics=("rmse", "crps", "bias", "spread", "skew", "kurt"),
                      vars=nb.ALL_VARS,
                      extra=["dep_b", "n_active_f_point", "spread_f_point_obs"])
df = nb.load_runs(RUNS, columns=COLS)

COMBOS = nb.combos_in(df)
A = nb.align(df, COMBOS)
print(A, "\n")
cov = A.coverage()
print(cov.to_string(index=False))

assert cov["n_points"].nunique() == 1, "combos do not share a common point set"
if cov["n_nan"].max() > 0:
    print(f"\nnote: {int((cov.n_nan > 0).sum())} combo(s) carry NaNs. skill_summary "
          f"drops them per combo and reports n_nan, so a combo with more NaNs is "
          f"scored on fewer points -- check n_nan before comparing medians.")

locs = sorted({round(float(c[3]), 3) for c in A.combos})
METHODS = sorted({(c[0], int(c[1])) for c in A.combos},
                 key=lambda t: (t[0] != "LETKF", t[0] != "AOEI", t[1]))
print(f"\nlocalization scales: {locs}\nmethods: {METHODS}")

## 0 · Cross-check against N3

This must pass before any number below means anything.

In [ ]:
head = nb.skill_summary(A, "rmse", "obs", "w")
try:
    got = nb.expect("N3_headline", head, N3_DIGEST)
    print(f"N3 <-> N4 CONSISTENT   digest = {got}")
except nb.ConsistencyError as e:
    print("CROSS-CHECK FAILED\n")
    if e.diff is not None:
        cols = [c for c in e.diff.columns
                if c.endswith(("_delta", "method", "ntemp", "loc", "red"))]
        print(e.diff[[c for c in e.diff.columns if c in cols]].to_string(index=False))
    raise

## 1 · Observation-space skill by method

TEnKF as a line against Nt; AOEI and LETKF as horizontal reference lines, since
they have no Nt. Left panel RMSE, right panel CRPS — **the same points, the same
reduction, only the metric changes.**

In [ ]:
SUM = pd.concat([nb.skill_summary(A, m, "obs", red)
                 for m in METRICS for red in nb.REDUCTIONS], ignore_index=True)

fig, axes = plt.subplots(2, 2, figsize=(nb.PAGE_W, nb.PAGE_W * 0.62), sharex=True)
for col, metric in enumerate(METRICS):
    for row, (field, lab) in enumerate([("frac_improved", "fraction of points improved"),
                                        ("median_skill", "median skill")]):
        ax = axes[row, col]
        for loc in locs:
            s = SUM[(SUM.metric == metric) & (SUM.red == "w") & np.isclose(SUM.loc_km, loc)]
            t = s[s.method == "TEnKF"].sort_values("ntemp")
            ax.plot(t.ntemp, t[field], "-o", ms=4, color=nb.loc_color(loc),
                    label=f"TEnKF, {nb.loc_label(loc)}")
            for meth, ls in (("LETKF", "--"), ("AOEI", ":")):
                r = s[s.method == meth]
                if len(r):
                    ax.axhline(float(r[field].iloc[0]), ls=ls, lw=1.1,
                               color=nb.loc_color(loc), alpha=0.75,
                               label=f"{meth}, {nb.loc_label(loc)}")
        ax.axhline(0.5 if field == "frac_improved" else 0.0, color="k", lw=0.8,
                   ls="--" if field == "frac_improved" else "-")
        ax.set_ylabel(lab, fontsize=8)
        if row == 0:
            ax.set_title(f"{nb.METRIC_LABELS[metric]}, loc-weighted", fontsize=9)
        if row == 1:
            ax.set_xlabel("tempering steps $N_t$")
            ax.set_xticks(sorted(t.ntemp.unique()))
axes[0, 0].legend(fontsize=6, ncol=2)
fig.suptitle("Method comparison in observation space  (>0 / >0.5 = analysis better)",
             y=1.01)
nb.save_fig(fig, "N4_fig1_method_ranking")
plt.show()

If the RMSE and CRPS panels rank the methods differently, that is N3 §3 showing up
at the aggregate level: a method that sharpens the ensemble too aggressively wins
on the mean-error metric and loses on the distributional one.

## 2 · Paired differences, not marginal means

Every method saw the same observations, so the honest comparison is per point:
`skill(method) − skill(baseline)` at each observation, then summarise *that*.
Comparing two marginal means computed over slightly different NaN sets is the
classic way two analyses of the same data end up disagreeing — and `align` plus
the `n_nan` column above are what stop it happening here.

In [ ]:
BASE = ("TEnKF", 1)

def paired(loc, metric="rmse", var="obs", red="w"):
    b = [c for c in A.combos if (c[0], int(c[1])) == BASE and np.isclose(float(c[3]), loc)][0]
    sb = nb.skill(A.frames[b], metric, var, red)
    out = {}
    for (m, nt) in METHODS:
        if (m, nt) == BASE:
            continue
        c = [k for k in A.combos
             if (k[0], int(k[1])) == (m, nt) and np.isclose(float(k[3]), loc)][0]
        out[(m, nt)] = (nb.skill(A.frames[c], metric, var, red) - sb).to_numpy(float)
    return out

fig, axes = plt.subplots(1, 2, figsize=(nb.PAGE_W, nb.PAGE_W * 0.38), sharey=True)
for ax, loc in zip(axes, locs):
    P = paired(loc)
    labs = [nb.combo_label(m, nt) for (m, nt) in P]
    data = [d[np.isfinite(d)] for d in P.values()]
    bp = ax.boxplot(data, labels=labs, showfliers=False, patch_artist=True,
                    medianprops=dict(color="k"))
    for patch, (m, nt) in zip(bp["boxes"], P):
        patch.set_facecolor(nb.combo_color(m, nt)); patch.set_alpha(0.75)
    for x, d in enumerate(data, start=1):
        ax.text(x, ax.get_ylim()[1], f"{100*(d > 0).mean():.0f}%",
                ha="center", va="bottom", fontsize=6)
    ax.axhline(0, color="k", lw=1.0)
    ax.set_title(f"{nb.loc_label(loc)}   (% = share of points beating {nb.combo_label(*BASE)})",
                 fontsize=8)
    ax.tick_params(axis="x", rotation=45)
axes[0].set_ylabel(f"paired RMSE skill vs {nb.combo_label(*BASE)} [dBZ]")
fig.suptitle("Per-point paired differences against the single-step baseline", y=1.04)
nb.save_fig(fig, "N4_fig3_paired")
plt.show()

In [ ]:
# LETKF and TEnKF Nt=1 are the same mathematics through different code paths.
print("LETKF vs TEnKF Nt=1 code-path check")
for loc in locs:
    try:
        cl = [c for c in A.combos if c[0] == "LETKF" and np.isclose(float(c[3]), loc)][0]
        ct = [c for c in A.combos if (c[0], int(c[1])) == ("TEnKF", 1)
              and np.isclose(float(c[3]), loc)][0]
    except IndexError:
        print(f"  {nb.loc_label(loc)}: one of the two combos is absent -- skipped")
        continue
    d = (nb.skill(A.frames[cl], "rmse", "obs", "w")
         - nb.skill(A.frames[ct], "rmse", "obs", "w")).abs()
    worst = float(np.nanmax(d))
    verdict = "PASS" if worst < nb.TOL["obs"] else "FAIL"
    print(f"  {nb.loc_label(loc)}: max |LETKF - TEnKF(1)| = {worst:.3e} dBZ   {verdict}")
print("\nIf PASS, 'LETKF' here is a code-path check rather than an independent method,")
print("and should be captioned that way. If FAIL, that is a finding worth chasing.")

## 3 · State variables, not only reflectivity

Only reflectivity is observed. The other seven variables move through their
cross-covariance with it, and nothing in observation space says whether they moved
the right way — the sweep's per-variable metrics are the only check.

Colour is `frac_improved`, which is unit-free, so `qg` (10⁻³ kg/kg) and `P`
(10⁴ Pa) share one scale. The annotation is the median skill in display units.
Variable order comes from `nbcommon.VARS`, so the ordering bug that mislabelled
every `qg`/`qr`/`qs` curve in the old notebooks cannot recur.

In [ ]:
LOC_SHOW = locs[0]
rows = []
for (m, nt) in METHODS:
    c = [k for k in A.combos if (k[0], int(k[1])) == (m, nt)
         and np.isclose(float(k[3]), LOC_SHOW)][0]
    sub = nb.Aligned({c: A.frames[c]}, [c], {c: A.nan_counts[c]}, 0)
    for v in nb.ALL_VARS:
        s = nb.skill_summary(sub, "rmse", v, "w").iloc[0]
        rows.append(dict(method=nb.combo_label(m, nt), var=v,
                         frac=s.frac_improved,
                         med=float(nb.disp(s.median_skill, v))))
MAT = pd.DataFrame(rows)

piv_f = MAT.pivot(index="method", columns="var", values="frac").reindex(
    index=[nb.combo_label(m, nt) for (m, nt) in METHODS], columns=nb.ALL_VARS)
piv_m = MAT.pivot(index="method", columns="var", values="med").reindex_like(piv_f)

fig, ax = plt.subplots(figsize=(nb.PAGE_W, nb.PAGE_W * 0.42))
im = ax.imshow(piv_f.to_numpy(), cmap=nb.CMAP_SKILL, vmin=0.0, vmax=1.0, aspect="auto")
ax.set_xticks(range(len(nb.ALL_VARS)))
ax.set_xticklabels([f"{nb.VAR_LABELS[v]}" for v in nb.ALL_VARS])
ax.set_yticks(range(len(piv_f.index))); ax.set_yticklabels(piv_f.index)
for r in range(piv_f.shape[0]):
    for c in range(piv_f.shape[1]):
        ax.text(c, r, f"{piv_m.iat[r, c]:.3g}", ha="center", va="center", fontsize=5.5,
                color="white" if abs(piv_f.iat[r, c] - 0.5) > 0.28 else "black")
plt.colorbar(im, ax=ax, shrink=0.8, label="fraction of points improved")
ax.set_title(f"Method x variable, {nb.loc_label(LOC_SHOW)}, loc-weighted RMSE\n"
             f"(annotation: median skill in display units)", fontsize=8)
ax.grid(False)
nb.save_fig(fig, "N4_fig4_method_by_variable")
plt.show()

In [ ]:
# The unobserved wind components: where tempering is most likely to misbehave.
fig, axes = plt.subplots(1, 3, figsize=(nb.PAGE_W, nb.PAGE_W * 0.30), sharey=True)
for ax, v in zip(axes, ["u", "v", "w"]):
    for loc in locs:
        s = pd.concat([nb.skill_summary(
                nb.Aligned({c: A.frames[c]}, [c], {c: 0}, 0), "rmse", v, "w")
            for c in A.combos if np.isclose(float(c[3]), loc)], ignore_index=True)
        t = s[s.method == "TEnKF"].sort_values("ntemp")
        ax.plot(t.ntemp, t.frac_improved, "-o", ms=4, color=nb.loc_color(loc),
                label=nb.loc_label(loc))
        for meth, ls in (("LETKF", "--"), ("AOEI", ":")):
            r = s[s.method == meth]
            if len(r):
                ax.axhline(float(r.frac_improved.iloc[0]), ls=ls, lw=1.0,
                           color=nb.loc_color(loc), alpha=0.7)
    ax.axhline(0.5, color="k", lw=0.8, ls="--")
    ax.set_title(f"{nb.VAR_LABELS[v]} (unobserved)", fontsize=8)
    ax.set_xlabel("tempering steps $N_t$")
axes[0].set_ylabel("fraction of points improved")
axes[0].legend(fontsize=7)
fig.suptitle("Unobserved wind components: updated only through cross-covariance", y=1.04)
nb.save_fig(fig, "N4_fig5_unobserved")
plt.show()

## 4 · Does the ranking depend on the reduction?

If a method is best at the observation point but not over the cutoff zone, then
"best method" is under-specified. Rank inversions between the three panels are
annotated rather than left for the reader to spot.

In [ ]:
fig, axes = plt.subplots(1, len(locs), figsize=(nb.PAGE_W, nb.PAGE_W * 0.36),
                         sharey=True)
inversions = []
for ax, loc in zip(np.atleast_1d(axes), locs):
    ranks = {}
    for red in nb.REDUCTIONS:
        s = SUM[(SUM.metric == "rmse") & (SUM.red == red) & np.isclose(SUM.loc_km, loc)].copy()
        s["lab"] = [nb.combo_label(m, n) for m, n in zip(s.method, s.ntemp)]
        s = s.sort_values("frac_improved", ascending=False).reset_index(drop=True)
        for rank, lab in enumerate(s.lab, start=1):
            ranks.setdefault(lab, []).append(rank)
    for lab, rr in ranks.items():
        meth = "TEnKF" if lab.startswith("TEnKF") else lab
        nt = int(lab.split("=")[-1]) if "=" in lab else 1
        ax.plot(range(len(nb.REDUCTIONS)), rr, "-o", ms=5,
                color=nb.combo_color(meth, nt), label=lab)
        ax.annotate(lab, (len(nb.REDUCTIONS) - 1, rr[-1]), fontsize=6,
                    xytext=(4, 0), textcoords="offset points", va="center")
        if len(set(rr)) > 1:
            inversions.append((loc, lab, rr))
    ax.set_xticks(range(len(nb.REDUCTIONS)))
    ax.set_xticklabels([nb.RED_LABELS[r] for r in nb.REDUCTIONS], fontsize=7)
    ax.invert_yaxis(); ax.set_title(nb.loc_label(loc), fontsize=8)
np.atleast_1d(axes)[0].set_ylabel("rank by fraction improved (1 = best)")
fig.suptitle("Does the method ranking survive changing the reduction?", y=1.03)
nb.save_fig(fig, "N4_fig6_rank_bump")
plt.show()

if inversions:
    print("RANK INVERSIONS -- these methods change position between reductions:")
    for loc, lab, rr in inversions:
        print(f"   {nb.loc_label(loc):10s} {lab:14s} ranks {rr} "
              f"across {list(nb.REDUCTIONS)}")
else:
    print("No rank inversions: the ordering is the same under all three reductions.")

## 5 · Multi-obs corroboration (optional)

The sweep assimilates each observation independently, so it cannot see
observation–observation interaction. If a multi-obs run exists, its domain-wide
`*_global_*` scalars are a coarse but independent check on the ranking above.
This section skips cleanly when no such run is present.

In [ ]:
if nb.has_multiobs(MO_TAG):
    mo = pd.concat([nb.mo_globals(p) for p in nb.multiobs_files(MO_TAG)],
                   ignore_index=True)
    print(mo.to_string(index=False))
    fig, ax = plt.subplots(figsize=(nb.COL_W * 1.7, nb.COL_W))
    for metric, mk in (("rmse", "o"), ("crps", "s")):
        s = mo[(mo.metric == metric) & (mo.var == "obs")].sort_values("ntemp")
        ax.plot(s.ntemp, s.skill, "-" + mk, label=nb.METRIC_LABELS[metric])
    ax.axhline(0, color="k", lw=0.8)
    ax.set_xlabel("tempering steps $N_t$")
    ax.set_ylabel("domain-wide skill [dBZ]  (>0 better)")
    ax.legend()
    nb.save_fig(fig, "N4_fig7_multiobs")
    plt.show()
else:
    print(f"no multi-obs run '{MO_TAG}' on disk -- section skipped.\n"
          f"The sweep-based conclusions above are unaffected; they simply cannot\n"
          f"speak to observation-observation interaction.")

## 6 · Conclusions, audited

Every statement below is re-derived from the data in the same execution. A claim
that no longer holds prints FAIL, so the prose cannot drift away from the figures
as the experiments are re-run.

In [ ]:
FULL = pd.concat([nb.skill_summary(A, m, v, red)
                  for m in METRICS for v in nb.ALL_VARS for red in nb.REDUCTIONS],
                 ignore_index=True)
nb.publish("N4_full", FULL)


def g(metric, var, red, method, ntemp, loc, field="frac_improved"):
    s = FULL[(FULL.metric == metric) & (FULL['var'] == var) & (FULL.red == red)
             & (FULL.method == method) & (FULL.ntemp == ntemp)
             & np.isclose(FULL.loc_km, loc)]
    return float(s[field].iloc[0]) if len(s) else np.nan


L0, L1 = locs[0], locs[-1]
CLAIMS = [
    ("Assimilation improves the majority of points at the tighter localization",
     g("rmse", "obs", "w", "TEnKF", 2, L0) > 0.5),
    ("Tempering (Nt=2) beats the single step (Nt=1) in observation space",
     g("rmse", "obs", "w", "TEnKF", 2, L0) > g("rmse", "obs", "w", "TEnKF", 1, L0)),
    ("The unweighted (cutoff-zone) score is no better than the point score",
     g("rmse", "obs", "u", "TEnKF", 2, L0) <= g("rmse", "obs", "point", "TEnKF", 2, L0)),
    ("CRPS and RMSE agree on whether Nt=2 beats Nt=1",
     (g("crps", "obs", "w", "TEnKF", 2, L0) > g("crps", "obs", "w", "TEnKF", 1, L0))
     == (g("rmse", "obs", "w", "TEnKF", 2, L0) > g("rmse", "obs", "w", "TEnKF", 1, L0))),
    ("The wider localization does not improve the observation-space hit rate",
     g("rmse", "obs", "w", "TEnKF", 2, L1) <= g("rmse", "obs", "w", "TEnKF", 2, L0)),
    ("Vertical wind w is improved less often than reflectivity",
     g("rmse", "w", "w", "TEnKF", 2, L0) < g("rmse", "obs", "w", "TEnKF", 2, L0)),
]

print("CLAIM AUDIT\n" + "=" * 78)
for text, ok in CLAIMS:
    print(f"  [{'PASS' if ok else 'FAIL'}]  {text}")
print("=" * 78)
print("A FAIL is not an error -- it means the sentence in the markdown above must be")
print("rewritten to match what the current experiments actually show.")